# End-to-End PDF RAG Lab with Qwen + FAISS\n
This notebook teaches a complete RAG lifecycle: PDF loading, metadata, chunking, FAISS, Qwen generation, evaluation, and MLOps thinking.

## 1. Install dependencies\n
Run this in Google Colab.

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers pypdf transformers accelerate pandas pytest\n

## 2. Create sample PDF\n
For teaching, we generate a small PDF so every student has the same dataset.

In [ ]:
# !pip install -q reportlab\nfrom reportlab.pdfgen import canvas\nfrom reportlab.lib.pagesizes import letter\nimport textwrap, os\n\nos.makedirs('data', exist_ok=True)\npdf_path = 'data/sample_policy.pdf'\nc = canvas.Canvas(pdf_path, pagesize=letter)\npages = [\n    ('Annual Leave Policy', 'Employees are entitled to 21 days of annual leave per year. Leave requests must be submitted in advance.'),\n    ('Working Hours', 'The normal working time is from 9 AM to 5 PM, Sunday to Thursday. Employees should record attendance daily.'),\n    ('Remote Work Policy', 'Employees may work remotely after receiving manager approval. Remote work should not affect productivity or communication.')\n]\nfor title, body in pages:\n    c.setFont('Helvetica-Bold', 18); c.drawString(72, 720, title)\n    c.setFont('Helvetica', 12); y = 680\n    for line in textwrap.wrap(body, width=80):\n        c.drawString(72, y, line); y -= 20\n    c.showPage()\nc.save()\npdf_path\n

## 3. Load PDF pages with metadata

In [ ]:
from langchain_community.document_loaders import PyPDFLoader\n\nloader = PyPDFLoader(pdf_path)\ndocs = loader.load()\nfor d in docs:\n    d.metadata['source'] = 'sample_policy.pdf'\n    d.metadata['page_number'] = d.metadata.get('page', 0) + 1\n    d.metadata['page_description'] = ' '.join(d.page_content.split()[:15])\n\nprint('Pages:', len(docs))\nprint(docs[0].metadata)\nprint(docs[0].page_content[:200])\n

## 4. Semantic-like chunking\n
We split by paragraphs/sentences first, not only fixed characters.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter\n\nsplitter = RecursiveCharacterTextSplitter(\n    chunk_size=500,\n    chunk_overlap=80,\n    separators=['\\n\\n', '\\n', '. ', '? ', '! ', ' ', '']\n)\nchunks = splitter.split_documents(docs)\nfor i, ch in enumerate(chunks):\n    ch.metadata['chunk_id'] = f\"{ch.metadata['source']}_p{ch.metadata['page_number']}_c{i}\"\n\nprint('Chunks:', len(chunks))\nprint(chunks[0].metadata)\n

## 5. Embeddings + FAISS

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings\nfrom langchain_community.vectorstores import FAISS\n\nembeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')\nvectorstore = FAISS.from_documents(chunks, embeddings)\nretriever = vectorstore.as_retriever(search_kwargs={'k': 3})\nprint('FAISS index ready')\n

## 6. Test retrieval and show page references

In [ ]:
query = 'How many annual leave days are allowed?'\nresults = retriever.invoke(query)\nfor r in results:\n    print('Page:', r.metadata['page_number'])\n    print('Source:', r.metadata['source'])\n    print(r.page_content[:250])\n    print('-'*60)\n

## 7. Load Qwen open-source LLM\n
For Colab CPU, use 0.5B. For GPU, students can try 1.5B.

In [ ]:
import torch\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, pipeline\n\nmodel_name = 'Qwen/Qwen2.5-0.5B-Instruct'\ntokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)\nmodel = AutoModelForCausalLM.from_pretrained(\n    model_name,\n    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\n    device_map='auto' if torch.cuda.is_available() else None,\n    trust_remote_code=True\n)\npipe = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=250, temperature=0.1, do_sample=False)\nprint('Qwen loaded')\n

## 8. Build RAG answer function

In [ ]:
def qwen_generate(prompt):\n    messages = [\n        {'role': 'system', 'content': 'You are a RAG assistant. Answer only from the provided context.'},\n        {'role': 'user', 'content': prompt}\n    ]\n    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    out = pipe(formatted)[0]['generated_text']\n    return out[len(formatted):].strip()\n\ndef answer_question(question):\n    docs = retriever.invoke(question)\n    context = '\\n\\n'.join([f\"Source={d.metadata['source']} Page={d.metadata['page_number']}\\n{d.page_content}\" for d in docs])\n    prompt = f\"\"\"\nUse ONLY this context to answer. If not found, say you do not know.\n\nContext:\n{context}\n\nQuestion: {question}\n\nReturn direct answer and sources.\n\"\"\"\n    return qwen_generate(prompt), docs\n\nanswer, srcs = answer_question('What is the normal working time?')\nprint(answer)\n

## 9. Retrieval evaluation\n
We check if the expected page appears in top-k results.

In [ ]:
import pandas as pd\n\ngolden = pd.DataFrame([\n    {'question': 'How many annual leave days are allowed?', 'expected_page': 1},\n    {'question': 'What is the normal working time?', 'expected_page': 2},\n    {'question': 'What should employees do before remote work?', 'expected_page': 3},\n])\n\nrows = []\nfor _, row in golden.iterrows():\n    docs_ret = retriever.invoke(row['question'])\n    pages = [d.metadata['page_number'] for d in docs_ret]\n    rows.append({'question': row['question'], 'expected_page': row['expected_page'], 'retrieved_pages': pages, 'hit': int(row['expected_page'] in pages)})\n\neval_df = pd.DataFrame(rows)\nprint(eval_df)\nprint('Recall@3 =', eval_df['hit'].mean())\n

## 10. Student tasks\n
1. Upload your own PDF.\n

2. Add OCR for scanned documents.\n

3. Change chunk_size from 500 to 800.\n

4. Compare Recall@3 before/after.\n

5. Add Streamlit UI.\n

6. Push the repo to GitHub and run CI/CD.